# GTF Exon skipping tool
**Obiettivo:** Rilevare gli eventi di *exon skipping* nell'annotazione genica del cromosoma 10 (UCSC, formato `GTF`).

Un evento di *exon skipping* si verifica quando due esoni consecutivi E1 ed E2 del trascritto condividono rispettivamente il sito di splicing al 5' con un esone E1' e il sito al 3' con un esone E2' del riferimento, e E1' ed E2' non sono consecutivi:
- **CE** (*cassette exon*): un solo esone saltato
- **ME** (*multiple-exon skipping*): più esoni saltati

**Nota:** Per ogni gene, il trascritto con più esoni è il *trascritto di riferimento*

**Input:** Il dataset `GTF` del cromosoma 10 e il file `gene-transcript-map.txt`, necessario per mappare l’identificatore di un trascritto al corrispondente gene

**Output:** Un record per evento: `(id del trascritto, id del trascritto di riferimento, tipo [, numero di esoni se ME])`.

## 1) Parametri in input e importazione delle librerie

In [ ]:
filename = "chr10.gtf"

In [ ]:
filename_map = "gene-transcript-map.txt"

In [ ]:
import re
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

## 2) Creazione di un generatore di *esoni* da un file `GTF`
**INPUT**:
- nome del file `GTF`

**OUTPUT**:
- tupla `(transcript_id, start, end, strand)` corrispondente ad un *esone*

In [ ]:
def read_and_parse_gtf_exons(gtf_filename):
    with open(gtf_filename) as f:
        for line in f:
            if line.startswith('#'):
                continue
            _, _, feature, start, end, _, fstrand, _, attrs = line.split('\t')
            if feature == 'exon':
                transcript_id = re.search(r'\btranscript_id\s+"(.+?)"', attrs).group(1)
                strand = 1 if fstrand == '+' else -1
                yield (transcript_id, int(start), int(end), strand)

## 3) Ottenimento della lista di esoni

In [ ]:
exons_list = [exon for exon in read_and_parse_gtf_exons(filename)]

In [ ]:
exons_list

## 4) Creazione del dizionario dei trascritti
- Il dizionario `transcripts_dict` associa a ogni trascritto un dizionario nidificato contenente la lista delle coordinate di tutti i suoi `esoni` e lo `strand` di appartenenza

Gli esoni vengono riordinati rispetto allo strand di appartenenza

In [ ]:
transcripts_dict = {}

for transcript_id, start, end, strand in exons_list:
    inner_dict = transcripts_dict.setdefault(transcript_id, {"exons": [], "strand": strand})
    inner_dict["exons"].append((start, end))

for transcript, inner_dict in transcripts_dict.items():
    inner_dict["exons"].sort(reverse=(transcripts_dict[transcript]["strand"] < 0), key=lambda x: x[0])

In [ ]:
transcripts_dict

## 5) Creazione del dizionario dei trascritti per gene e scelta del trascritto di riferimento
Il dizionario `gene_transcripts_dict` associa a ogni gene un dizionario nidificato contenente l'*ID del trascritto di riferimento* e il set degli *ID di tutti i trascritti* associati a quel gene.

Utilizzo `transcript_lookup` per mappare i transcript ID senza versione ai transcript ID con versione, per consistenza con il file  `gene-transcript-map.txt`

**Nota:** a parità di numero di esoni, viene scelto come riferimento il trascritto con id minore

In [ ]:
transcript_lookup = {}
for transcript_id in transcripts_dict:
    transcript_lookup[transcript_id.split('.')[0]] = transcript_id

In [ ]:
gene_transcripts_dict = {}

with open(filename_map) as f:
    next(f, None)
    for line in f:
        gene_id, transcript_id = line.strip().split()
        transcript_id = transcript_lookup.get(transcript_id)
        if transcript_id is not None:
            gene_transcripts_dict.setdefault(gene_id, {'ref': None, 'transcripts': set()})['transcripts'].add(transcript_id)

In [ ]:
for gene, gene_dict in gene_transcripts_dict.items():
    transcripts = gene_dict['transcripts']
    gene_dict['ref'] = min(transcripts, key=lambda t: (-len(transcripts_dict[t]["exons"]), t))

In [ ]:
gene_transcripts_dict

## 6) Definizione della funzione per il rilevamento di eventi di *exon skipping*
**INPUT**:
- *ID del trascritto alternativo*
- *ID del trascritto di riferimento*
- *strand* di appartenenza dei trascritti
- lista degli *esoni del trascritto di riferimento* (**ordinata**)
- lista degli *esoni del trascritto alternativo* (**ordinata**)


**OUTPUT**:
- lista di tuple `(id_tr, id_ref, event_type[, count])` corrispondenti a eventi di *exon skipping*

In [ ]:
# d: indice della coordinata che identifica il sito di splicing al 5' dell'esone (donor)
# a: indice della coordinata che identifica il sito di splicing al 3' dell'esone (acceptor)
# sr: indice da cui riprende la scansione del riferimento, aggiornato a ogni coppia
# head: l'indice dell'esone E1' nel trascritto di riferimento che condivide il sito di splicing al 5' con l'esone E1 del trascritto analizzato
def exon_skipping_finder(id_tr, id_ref, strand, ref, tr):
    d, a = (1, 0) if strand > 0 else (0, 1) 
    events = []
    sr = 0
    for e1, e2 in zip(tr, tr[1:]):
        head = None 
        for i, eR in enumerate(ref[sr:], start=sr):
            if head is None:
                if eR[d] == e1[d]:
                    head = i
                elif strand * eR[d] > strand * e1[d]:
                    sr = i
                    break
            elif eR[a] == e2[a]:
                count = i - head - 1
                if count > 0:
                    record = (id_tr, id_ref, "ME", count) if count > 1 else (id_tr, id_ref, "CE")
                    events.append(record)
                sr = i
                break
            elif strand * eR[a] > strand * e2[a]:
                sr = i - 1
                break 
        else:
            return events
    return events

## 7) Ricerca degli eventi di exon skipping relativi allo specifico dataset 


In [ ]:
events = []
for gene, gene_dict in gene_transcripts_dict.items():
    reference = gene_dict['ref']
    strand = transcripts_dict[reference]["strand"]
    for transcript in gene_dict['transcripts']:
        if transcript == reference:
            continue
        events_per_transcript = exon_skipping_finder(transcript,
                                      reference,
                                      strand,
                                      transcripts_dict[reference]["exons"],
                                      transcripts_dict[transcript]["exons"]
                                     )   
        events.extend(events_per_transcript)

In [ ]:
print(*events, sep='\n')

## 8) Interpretazione dei risultati
Possiamo notare che:
- Gli eventi di *exon skipping* più frequenti sono di tipo **CE** e la frequenza degli eventi diminuisce progressivamente all'aumentare del numero di esoni esclusi
    - Infatti, biologicamente è più raro saltare molti esoni consecutivi rispetto a saltarne uno solo
- Oltre la metà dei geni multi-isoforma presenta eventi di *exon skipping*
- La differenza tra media e mediana suggerisce una distribuzione asimmetrica, in cui pochi geni concentrano un numero elevato di eventi rispetto alla maggioranza

In [ ]:
n_genes_multi = sum(len(gene_dict['transcripts']) > 1 for gene_dict in gene_transcripts_dict.values())
print(f"Numero di geni multi-isoforma analizzati: {n_genes_multi}")

types = Counter(e[2] for e in events)
print(f"Numero di eventi di exon skipping: {len(events)} -> " + ", ".join(f"{t}: {n} ({n/len(events):.0%})" for t, n in types.items()))

events_per_gene = Counter(e[1] for e in events)

counts = [events_per_gene.get(gene_dict['ref'], 0)
          for gene_dict in gene_transcripts_dict.values()
          if len(gene_dict['transcripts']) > 1]

n_genes_with_events = sum(c > 0 for c in counts)
print(f"Geni multi-isoforma con almeno 1 evento: {n_genes_with_events} ({n_genes_with_events/n_genes_multi:.0%})")
print(f"Numero medio di eventi per gene multi-isoforma: {np.mean(counts):.2f}")
print(f"Mediana eventi per gene multi-isoforma: {np.median(counts)}")

In [ ]:
sizes = Counter(e[3] if len(e) == 4 else 1 for e in events) 
sorted_sizes = sorted(sizes.items())
x, y = [str(exons) for exons, events in sorted_sizes], [events for exons, events in sorted_sizes]      
bars = plt.bar(x, y, color='blue', width = 0.5)
plt.yscale('log')
plt.bar_label(bars, fontsize=8) 
plt.xlabel('Nr esoni saltati')
plt.ylabel('Nr eventi')
plt.title('Distribuzione esoni saltati per evento')
plt.show()